# Validação do peso ótimo em coil_temps e ipc_comp

Compara o peso ótimo local de cada área com o peso global, escolhido com as sete áreas juntas.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_treinamento"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os

import pandas as pd
import numpy as np

PASTA_DADOS = "colabs_S3"

BASES = [
    ".", "..", os.path.join("..", ".."), os.path.join("..", "..", ".."), "/content",
    "notebooks", os.path.join("..", "notebooks"),
    os.path.join("notebooks", PASTA_DADOS),
]

def resolve(pasta, arquivo):
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas)
    )

In [ ]:
previsoes = pd.read_csv(resolve(PASTA_DADOS, "previsoes_rf_gb.csv"))
limites = pd.read_csv(resolve(PASTA_DADOS, "limites_ativos.csv")).set_index("tc_name")
previsoes.shape

In [ ]:
MARGEM_KPI = 0.10   # margem de erro acordada no TAPI


def _valida(y_real, y_previsto):
    """Converte para vetores e recusa entradas que produziriam metrica sem sentido.

    A conversao para numpy e necessaria porque a subtracao entre duas Series do pandas
    alinha pelos indices, e nao pela posicao. Uma Series com indice original e outra
    com indice reiniciado produziriam diferencas entre pares errados, em silencio.

    O alvo precisa ser estritamente positivo. A restricao e legitima neste projeto,
    pois trata-se de temperatura absoluta em graus Celsius, cujo minimo na base e de
    27,69 graus. Ela existe para transformar em erro visivel o engano de passar valores
    padronizados, caso em que o alvo se aproximaria de zero e o erro percentual
    explodiria em vez de acusar o problema.
    """
    y_real = np.asarray(y_real, dtype=float).ravel()
    y_previsto = np.asarray(y_previsto, dtype=float).ravel()

    if y_real.shape != y_previsto.shape:
        raise ValueError(f"dimensoes diferentes: {y_real.shape} e {y_previsto.shape}")
    if y_real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(y_real).any():
        raise ValueError("ha valores ausentes em y_real; a mascara deve ser aplicada antes")
    if np.isnan(y_previsto).any():
        raise ValueError("ha valores ausentes em y_previsto")
    if (y_real <= 0).any():
        raise ValueError(
            "y_real contem valor nao positivo; o erro percentual exige temperatura "
            "absoluta em graus Celsius, e nao valores padronizados")

    return y_real, y_previsto


def avaliar(y_real, y_previsto, limite=None):
    """Metricas de erro de um componente, com o alvo em graus Celsius.

    Sempre devolvidas:

    n            numero de observacoes que sustentam as metricas
    MAE          erro absoluto medio, na unidade do alvo
    RMSE         raiz do erro quadratico medio, na mesma unidade
    R2           fracao da variancia do alvo explicada pelo modelo
    vies         erro medio com sinal, positivo quando o modelo superestima
    RMSE/MAE     razao entre as duas, indicando concentracao dos erros
    MAPE_%       erro percentual absoluto medio
    WAPE_%       erro absoluto total dividido pela soma dos alvos
    dentro_10_%  proporcao de predicoes dentro da margem de 10 por cento do TAPI

    Devolvidas apenas quando um limite de seguranca e informado:

    limite_C          o proprio limite, para conferencia
    n_reprova_real    ensaios em que o componente de fato atinge o limite
    acerto_decisao_%  concordancia entre a decisao real e a prevista
    falso_aprovado    ensaios reprovados na realidade e aprovados pelo modelo
    falso_reprovado   ensaios aprovados na realidade e reprovados pelo modelo

    O R2 sai ausente quando o alvo assume um unico valor, situacao em que ele e
    indefinido e em que a implementacao do scikit-learn devolve zero, valor que se
    confunde com um resultado legitimo.
    """
    y_real, y_previsto = _valida(y_real, y_previsto)

    erro = y_previsto - y_real
    mae = float(np.mean(np.abs(erro)))

    # A raiz e calculada explicitamente: o parametro squared do mean_squared_error foi
    # removido do scikit-learn na versao 1.9, e o root_mean_squared_error so existe a
    # partir da 1.4. O calculo direto funciona em qualquer versao.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(y_real).size < 2:
        r2 = float("nan")
    else:
        ss_res = float(np.sum(erro ** 2))
        ss_tot = float(np.sum((y_real - np.mean(y_real)) ** 2))
        r2 = 1.0 - ss_res / ss_tot

    relativo = np.abs(erro) / y_real

    resultado = {
        "n": int(y_real.size),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "vies": float(np.mean(erro)),
        "RMSE/MAE": (rmse / mae) if mae > 0 else float("nan"),
        "MAPE_%": 100.0 * float(np.mean(relativo)),
        "WAPE_%": 100.0 * float(np.sum(np.abs(erro)) / np.sum(y_real)),
        "dentro_10_%": 100.0 * float(np.mean(relativo <= MARGEM_KPI)),
    }

    if limite is not None and np.isfinite(limite):
        reprova_real = y_real >= limite
        reprova_prevista = y_previsto >= limite
        resultado.update({
            "limite_C": float(limite),
            "n_reprova_real": int(reprova_real.sum()),
            "acerto_decisao_%": 100.0 * float(np.mean(reprova_real == reprova_prevista)),
            "falso_aprovado": int(np.sum(reprova_real & ~reprova_prevista)),
            "falso_reprovado": int(np.sum(~reprova_real & reprova_prevista)),
        })

    return resultado


def consolidar(tabela, grupo="dataset",
               metricas=("MAE", "RMSE", "R2", "vies", "MAPE_%", "WAPE_%", "dentro_10_%")):
    """Consolida metricas por componente em metricas por area.

    A macro-media atribui o mesmo peso a cada componente e responde se o modelo funciona
    bem em todas as partes do produto. A micro-media pondera pelo numero de observacoes
    e responde qual o erro tipico de uma leitura qualquer.

    O R2 nao e comparavel entre areas, pois seu denominador e a variancia do alvo de
    cada area. Ele indica se o modelo supera a media dentro da propria area.
    """
    if grupo not in tabela.columns:
        raise ValueError(f"coluna de agrupamento ausente: {grupo}")
    if "n" not in tabela.columns:
        raise ValueError("a tabela precisa da coluna n para a micro-media")

    metricas = [m for m in metricas if m in tabela.columns]
    linhas = []

    for nome, bloco in tabela.groupby(grupo):
        linha = {grupo: nome, "componentes": len(bloco), "n_total": int(bloco["n"].sum())}
        for m in metricas:
            valores = bloco[m]
            validos = valores.notna()
            linha[f"{m}_macro"] = (float(valores[validos].mean())
                                   if validos.any() else float("nan"))
            if validos.any() and bloco.loc[validos, "n"].sum() > 0:
                linha[f"{m}_micro"] = float(
                    np.average(valores[validos], weights=bloco.loc[validos, "n"]))
            else:
                linha[f"{m}_micro"] = float("nan")

        # As metricas de decisao sao contagens, e por isso somadas em vez de promediadas
        for contagem in ("n_reprova_real", "falso_aprovado", "falso_reprovado"):
            if contagem in bloco.columns:
                linha[contagem] = int(bloco[contagem].fillna(0).sum())
        linhas.append(linha)

    return pd.DataFrame(linhas).sort_values(grupo).reset_index(drop=True)

In [ ]:
def mesclar(rf, gb, peso=0.5):
    """Combinacao ponderada das previsoes; peso e o do rf e (1 - peso) o do gb."""
    if not 0 <= peso <= 1:
        raise ValueError(f"peso fora de [0, 1]: {peso}")
    rf = np.asarray(rf, dtype=float)
    gb = np.asarray(gb, dtype=float)
    if rf.shape != gb.shape:
        raise ValueError(f"dimensoes diferentes: {rf.shape} e {gb.shape}")
    return peso * rf + (1 - peso) * gb

In [ ]:
AREAS = ["coil_temps", "ipc_comp"]
PESOS = np.round(np.linspace(0, 1, 21), 2)


def limite_de(col):
    lim = limites["limite_num"].get(col)
    return float(lim) if lim is not None and pd.notna(lim) else None


# a varredura cobre todas as areas porque o peso global depende delas
linhas = []
for (sufixo, area, col), p in previsoes.groupby(["sufixo", "area", "componente"], sort=False):
    limite = limite_de(col)
    for peso in PESOS:
        linhas.append({"sufixo": sufixo, "area": area, "componente": col, "peso": peso,
                       **avaliar(p["real"], mesclar(p["rf"], p["gb"], peso), limite=limite)})

metricas = pd.DataFrame(linhas)
metricas.shape

In [ ]:
# consolidar por sufixo da a macro-media sobre os 74 componentes das sete areas
blocos = [consolidar(bloco, grupo="sufixo").assign(peso=peso)
          for peso, bloco in metricas.groupby("peso", sort=False)]
por_sufixo = pd.concat(blocos, ignore_index=True)

peso_global = (por_sufixo.loc[por_sufixo.groupby("sufixo")["MAE_macro"].idxmin(),
                              ["sufixo", "peso", "MAE_macro"]]
               .rename(columns={"peso": "peso_global", "MAE_macro": "MAE_global_todas"})
               .set_index("sufixo"))
peso_global

In [ ]:
blocos = [consolidar(bloco, grupo="area").assign(sufixo=sufixo, peso=peso)
          for (sufixo, peso), bloco in metricas.groupby(["sufixo", "peso"], sort=False)]
por_area = pd.concat(blocos, ignore_index=True)
por_area = por_area[por_area["area"].isin(AREAS)]

CHAVE = ["sufixo", "area"]
mae = por_area.pivot(index=CHAVE, columns="peso", values="MAE_macro")

# idxmin devolve o primeiro minimo, ou seja o menor peso em caso de empate
comparacao = pd.DataFrame({"peso_local": mae.idxmin(axis=1), "MAE_local": mae.min(axis=1)})
comparacao["peso_global"] = peso_global["peso_global"].reindex(
    comparacao.index.get_level_values("sufixo")).to_numpy()
comparacao["MAE_global"] = [mae.loc[chave, peso] for chave, peso in comparacao["peso_global"].items()]
comparacao["MAE_rf"] = mae[1.0]
comparacao["MAE_gb"] = mae[0.0]
comparacao.round(3)

In [ ]:
import json

with open(resolve("colabs_S3/preparacao_testes", "particao_treino_teste.json"), encoding="utf-8") as f:
    particao = json.load(f)

# os run_idx de teste ficam com dobra None; o assert garante que nenhum entrou nas previsoes
DOBRA = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
previsoes["dobra"] = previsoes["run_idx"].map(DOBRA)
assert previsoes["dobra"].notna().all()
assert not set(previsoes["run_idx"]) & set(particao["run_idx_teste"])
previsoes["dobra"] = previsoes["dobra"].astype(int)
previsoes["dobra"].nunique()

In [ ]:
comparacao["individual"] = np.where(comparacao["MAE_rf"] <= comparacao["MAE_gb"], "rf", "gb")

linhas = []
for (sufixo, area), r in comparacao.iterrows():
    p = previsoes[(previsoes["sufixo"] == sufixo) & (previsoes["area"] == area)]
    for (dobra, col), q in p.groupby(["dobra", "componente"], sort=False):
        configs = {"otimo": mesclar(q["rf"], q["gb"], r["peso_local"]), "individual": q[r["individual"]]}
        for config, previsto in configs.items():
            linhas.append({"sufixo": sufixo, "area": area, "dobra": dobra, "componente": col,
                           "config": config, **avaliar(q["real"], previsto)})

por_componente = pd.DataFrame(linhas)
blocos = [consolidar(bloco, grupo="dobra").assign(sufixo=sufixo, area=area, config=config)
          for (sufixo, area, config), bloco in por_componente.groupby(["sufixo", "area", "config"], sort=False)]
mae_dobra = (pd.concat(blocos)
             .pivot(index=["sufixo", "area", "dobra"], columns="config", values="MAE_macro")
             [["otimo", "individual"]])
# positivo quando o peso otimo erra menos que o individual naquela dobra
mae_dobra["diferenca"] = mae_dobra["individual"] - mae_dobra["otimo"]
mae_dobra.shape

In [ ]:
def _sem_empate(d):
    return d[~np.isclose(d, 0)]


resumo = mae_dobra.groupby(CHAVE)["diferenca"].agg(
    dobras="size",
    otimo_melhor=lambda d: int((_sem_empate(d) > 0).sum()),
    empates=lambda d: int(np.isclose(d, 0).sum()),
    individual_melhor=lambda d: int((_sem_empate(d) < 0).sum()),
    diferenca_media="mean",
    diferenca_minima="min",
)
resumo = comparacao[["peso_local", "individual"]].join(resumo)
resumo.round(3)

In [ ]:
# uma coluna por sufixo, area e configuracao; uma linha por dobra
lista = (mae_dobra.unstack(["sufixo", "area"])
         .reorder_levels(["sufixo", "area", "config"], axis=1).sort_index(axis=1)
         .reindex(columns=["otimo", "individual", "diferenca"], level="config"))
lista.round(2)

In [ ]:
# Registro versionado da validacao, ao lado dos outros relatorios de documents/extras
def pasta_documentos():
    for base in BASES + [os.path.join("..", "..", "..", "..")]:
        pasta = os.path.join(base, "documents", "extras", "relatorios_sprint_4")
        if os.path.isdir(pasta):
            return pasta
    raise FileNotFoundError("pasta documents/extras/relatorios_sprint_4 nao encontrada a partir de " + os.path.abspath("."))


def _num(v, sinal=False):
    if pd.isna(v):
        return "–"
    if sinal and not np.isclose(v, 0):
        return f"{v:+.2f}".replace(".", ",")
    return f"{v:.2f}".replace(".", ",")


def _tabela(cabecalho, linhas, texto_cols=2):
    return ["| " + " | ".join(cabecalho) + " |",
            "| " + " | ".join(["---"] * texto_cols + ["---:"] * (len(cabecalho) - texto_cols)) + " |",
            *linhas]


cab_pesos = ["Sufixo", "Área", "Peso local", "MAE peso local", "Peso global", "MAE peso global", "MAE rf", "MAE gb"]
linhas_pesos = [
    "| " + " | ".join([sufixo, area] + [_num(r[c]) for c in
                       ("peso_local", "MAE_local", "peso_global", "MAE_global", "MAE_rf", "MAE_gb")]) + " |"
    for (sufixo, area), r in comparacao.iterrows()
]

cab_dobras = ["Sufixo", "Área", "Peso local", "Individual", "Dobras", "Peso local melhor", "Empates",
              "Individual melhor", "Diferença média", "Diferença mínima"]
linhas_dobras = [
    "| " + " | ".join([sufixo, area, _num(r["peso_local"]), r["individual"], str(int(r["dobras"])),
                       str(int(r["otimo_melhor"])), str(int(r["empates"])), str(int(r["individual_melhor"])),
                       _num(r["diferenca_media"], sinal=True), _num(r["diferenca_minima"], sinal=True)]) + " |"
    for (sufixo, area), r in resumo.iterrows()
]

texto = "\n".join([
    "# Validação do peso ótimo da mesclagem em coil_temps e ipc_comp",
    "",
    "Verificação do peso ótimo da mesclagem RF e GB nas áreas coil_temps e ipc_comp, a partir da varredura "
    "de pesos (`varredura_pesos_mesclagem.md`). O peso global é o peso único de menor MAE quando os 74 "
    "componentes das sete áreas são avaliados juntos, calculado por sufixo com a mesma média macro do "
    "baseline, e é comparado ao peso ótimo local de cada área. A verificação por dobra recalcula o MAE da "
    "área em cada uma das 17 dobras leave-one-group-out da validação, as mesmas do baseline "
    "(`baseline_metricas_por_area.md`), comparando a mesclagem no peso local com o melhor modelo individual "
    "da área; o conjunto de teste reservado não entra em nenhuma etapa. Gerado por "
    "`notebooks/colabs_S4/testes_extras/testes_treinamento/validacao_peso_otimo.ipynb`.",
    "",
    "## Peso local, peso global e MAE",
    "",
    "Médias macro (peso igual por componente). O peso é o do rf; o do gb é o complemento.",
    "",
    *_tabela(cab_pesos, linhas_pesos),
    "",
    "## Verificação por dobra",
    "",
    "MAE da área em cada dobra, com a mesclagem no peso local contra o melhor modelo individual da área. "
    "A diferença é o MAE do individual menos o da mesclagem, em °C; positiva quando a mesclagem erra menos. "
    "Empate é diferença nula.",
    "",
    *_tabela(cab_dobras, linhas_dobras, texto_cols=4),
    "",
    # texto fixo, escrito a partir das tabelas acima; conferir se PESOS ou as previsoes mudarem
    "Em ipc_comp o peso ótimo local é 0,00 nos dois sufixos, ou seja a mesclagem ótima é o próprio gb, e por "
    "isso as 17 dobras empatam por construção. Em coil_temps o ganho médio por dobra é positivo, 0,37 °C no "
    "`_13cat` e 0,21 °C no `_perfil`, mas a vitória por dobra é dividida, 10 a 7 e 8 a 9, e a média é "
    "puxada por poucas dobras com diferença grande, como a dobra 9 (+3,21 °C e +1,86 °C) e a dobra 15 "
    "(+2,16 °C e +0,87 °C), enquanto nas demais a diferença fica em torno de ±0,5 °C. Nas piores dobras a "
    "mesclagem perde por 0,92 °C e 0,40 °C. Por isso o ganho da mesclagem em coil_temps não se sustenta de "
    "forma consistente entre as dobras.",
    "",
])

caminho_md = os.path.join(pasta_documentos(), "validacao_peso_otimo.md")
with open(caminho_md, "w", encoding="utf-8", newline="\n") as f:
    f.write(texto)
caminho_md